In [22]:
import nltk
import string
from nltk.corpus import gutenberg
from nltk.corpus import stopwords
from gensim.models import Word2Vec
from scipy.stats import spearmanr

nltk.download('gutenberg')
nltk.download('punkt')
nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package gutenberg to
[nltk_data]     C:\Users\LiuYe\AppData\Roaming\nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\LiuYe\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\LiuYe\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [16]:
sentences: list[list[str]] = gutenberg.sents()
print(len(sentences))
print(sentences[0])

98552
['[', 'Emma', 'by', 'Jane', 'Austen', '1816', ']']


In [17]:
processed_sentences = []

for sent in sentences:
    new_sent = []
    for word in sent:
        word = word.lower()

        if word in string.punctuation:
            continue

        if word in stop_words:
            continue

        new_sent.append(word)

    if len(new_sent) > 0:
        processed_sentences.append(new_sent)

print(processed_sentences[0])

['emma', 'jane', 'austen', '1816']


In [18]:
# CBOW（sg=0）
model_cbow = Word2Vec(
    sentences=processed_sentences,
    vector_size=100,
    window=5,
    min_count=5,
    workers=4,
    sg=0,  # CBOW
    epochs=10
)

# Skip-gram（sg=1）
model_sg = Word2Vec(
    sentences=processed_sentences,
    vector_size=100,
    window=5,
    min_count=5,
    workers=4,
    sg=1,  # Skip-gram
    epochs=10
)
model_cbow.save("word2vec_cbow.model")
model_sg.save("word2vec_sg.model")

print(f"CBOW vocabulary size: {len(model_cbow.wv.key_to_index)}")
print(f"Skip-gram vocabulary size: {len(model_sg.wv.key_to_index)}")

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


CBOW vocabulary size: 15579
Skip-gram vocabulary size: 15579


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


In [20]:
print("CBOW:")
print(model_cbow.wv.most_similar("whale", topn=10))

print("\nSkip-gram:")
print(model_sg.wv.most_similar("whale", topn=10))

CBOW:
[('whales', 0.7558427453041077), ('sperm', 0.7440491318702698), ('fishery', 0.6760026812553406), ('leviathan', 0.6689490079879761), ('whalemen', 0.6406360864639282), ('chase', 0.6396847367286682), ('dick', 0.628677248954773), ('monster', 0.6249414682388306), ('angles', 0.6244227886199951), ('greenland', 0.6182026863098145)]

Skip-gram:
[('sperm', 0.8160043954849243), ('whales', 0.6681483387947083), ('alongside', 0.6366416811943054), ('fishery', 0.6292400360107422), ('greenland', 0.6230142116546631), ('whalemen', 0.5940989255905151), ('leviathan', 0.5918657183647156), ('hump', 0.5902855396270752), ('vicinity', 0.5814931988716125), (').--', 0.57691490650177)]


In [25]:
test_words = ['car', 'beautiful', 'play', 'computer']
for word in test_words:
    if word in model_cbow.wv.key_to_index:
        print(f"CBOW - Similar words to '{word}':")
        similar = model_cbow.wv.most_similar(word, topn=10)
        for word, score in similar:
            print(f"{word}: {score:.4f}")
        print('='*50)
    else:
        print(f"The word '{word}' is not in our literary corpus vocabulary")

CBOW - Similar words to 'car':
cab: 0.9016
heavily: 0.8951
bows: 0.8869
backwards: 0.8831
across: 0.8825
lawn: 0.8715
tashtego: 0.8707
slowly: 0.8688
motor: 0.8679
stern: 0.8667
CBOW - Similar words to 'beautiful':
lovely: 0.8014
grown: 0.7012
healthy: 0.6919
handsome: 0.6895
modest: 0.6810
delicate: 0.6791
gay: 0.6759
curious: 0.6700
growing: 0.6649
splendid: 0.6608
CBOW - Similar words to 'play':
welcome: 0.7180
merry: 0.7054
try: 0.6718
nice: 0.6680
bee: 0.6477
likes: 0.6389
maybe: 0.6357
follow: 0.6354
?--: 0.6338
tune: 0.6296
The word 'computer' is not in our literary corpus vocabulary


In [27]:
similarity_pairs = [
    ("car", "car", 9),
    ("computer", "laptop", 8),
    ("sun", "moon", 6),
    ("cat", "dog", 7),
    ("joy", "happiness", 8),
    ("house", "building", 7),
    ("fast", "fast", 8),
    ("red", "scarlet", 9),
    ("good", "bad", 4)
]

cosine_similarities = []
human_scores = []
for w1, w2, score in similarity_pairs:
    if w1 in model_cbow.wv.key_to_index and w2 in model_cbow.wv.key_to_index:
        cosine_sim = model_cbow.wv.similarity(w1, w2)
        cosine_similarities.append(cosine_sim)
        human_scores.append(score)
        print(
            f"Similarity between '{w1}' and '{w2}': {cosine_sim:.4f} (Human score: {score})")
res = spearmanr(cosine_similarities, human_scores)
print(f"\nSpearman: {res.statistic:.4f}, p_value: {res.pvalue:.4f}")

Similarity between 'car' and 'car': 1.0000 (Human score: 9)
Similarity between 'sun' and 'moon': 0.8633 (Human score: 6)
Similarity between 'cat' and 'dog': 0.6629 (Human score: 7)
Similarity between 'joy' and 'happiness': 0.4187 (Human score: 8)
Similarity between 'house' and 'building': 0.3482 (Human score: 7)
Similarity between 'fast' and 'fast': 1.0000 (Human score: 8)
Similarity between 'red' and 'scarlet': 0.7637 (Human score: 9)
Similarity between 'good' and 'bad': 0.5758 (Human score: 4)

Spearman: 0.4025, p_value: 0.3229


In [28]:
def test_identity_analogy(a, b):
    result = model_cbow.wv.most_similar(
        positive=[a, b], negative=[a], topn=1)
    print(
        f"{a:} - {a:} + {b:} === {result[0][0]:} (sim: {result[0][1]:.4f})")

test_identity_analogy('king', 'queen')
test_identity_analogy('man', 'doctor')
test_identity_analogy('eat', 'apple')
test_identity_analogy('river', 'bank')

king - king + queen === haman (sim: 0.6629)
man - man + doctor === gravely (sim: 0.8814)
eat - eat + apple === roots (sim: 0.9008)
river - river + bank === yard (sim: 0.8964)


### 1. Qualitative Evaluation of CBOW Model
For the word "car", the most similar words included "cab", "motor", and "stern", suggesting the model associated vehicles with maritime or horse-drawn contexts—again reflecting the literary nature of the corpus.

For "beautiful", the model returned reasonable synonyms such as "lovely", "handsome", and "splendid".

For "play", the results were less coherent, including "merry", "welcome", and even punctuation-like tokens (e.g., *"?--"*), indicating noise or limited contextual diversity for certain words.

### 2. Correlation with Human Judgment
A Spearman rank correlation was calculated between CBOW cosine similarities and human-assigned similarity scores for nine word pairs.

The correlation coefficient was 0.4025 with a p-value of 0.3229, which is not statistically significant (p > 0.05). This suggests that the CBOW model’s semantic similarity judgments do not strongly align with human intuition, possibly due to the small and domain-specific evaluation set or the inherent differences between literary co-occurrence statistics and human semantic perception.

### 3. Analogy Reasoning
The analogy tests (e.g., king – king + queen) did not produce meaningful results. This indicates the CBOW model fails to capture valid semantic similarity and meaningful word relationships in this task. The poor performance is likely caused by inadequate training data, limited corpus size, or insufficient training iterations, resulting in low-quality word embeddings that cannot represent genuine lexical semantics.